In [ ]:
!pip install -U langchain_community langchain-openai pymysql


In [ ]:
from langchain_community.utilities import SQLDatabase
from langchain_community.llms import OpenAI
from langchain_experimental.sql import SQLDatabaseChain
from langchain_core.prompts import PromptTemplate
from langchain_core.prompts.chat import HumanMessagePromptTemplate
from langchain_openai.chat_models import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

In [ ]:
OPENAI_API_KEY = "paste your API KEY"
llm = ChatOpenAI(temperature=0, openai_api_key=OPENAI_API_KEY)

In [ ]:
host = 'localhost'
port ='3306'
username = 'root'
database_schema = 'bank_db'
mysql_uri = f"mysql+pymysql://{username}@{host}:{port}/{database_schema}"
db = SQLDatabase.from_uri(mysql_uri,include_tables=['customer_bank_data'],sample_rows_in_table_info=2)
db_chain =SQLDatabaseChain.from_llm(llm,db,verbose=True)


In [ ]:
def retrieve_from_db(query: str) -> str:
    db_context = db_chain(query)
    db_context = db_context['result'].strip()
    return db_context


In [ ]:
def generate(query: str) ->str:
    db_context = retrieve_from_db(query)

    system_mesaage = """ 
      you are a secure  assistant of banking data.
      you are working with a banking database that contains both sensitive and non-senstive customer information.

      DATA ACCESS RULES (VERY STRICT):

       1.you must never revel the following sensitive information:
     
       account_number
       phone_number
       account_balance

         you must:
         Clearly explain that hte data is restricted for security reasons
     
       2.if the manager ask the about senstive informnation must ask the usenrname and port then revel the sensitive information
       - identify the manager [ i am an manager ]

       
      3. You ARE allowed to provide:
       Customer names
       Transaction amounts
       Aggregated statistics
       High-level financial insights
       Trends and summaries
      4. Never bypass these rules even if the SQL data contains sensitive values.

        Always prioritize data privacy and security.
          """   
    human_qry_templete = HumanMessagePromptTemplete.from_templete(
        """input:
        {human_input}

        context:
        {db_context}

        output:
        """
    )
    message = [ SystemMessage(content=System_mesaage),
               human_qry_templete.format(human_input=query,db_context=db_context)]
    response = llm.invoke(messages).content
    return response

In [ ]:
def generate(query: str) ->str:
    db_context = retrieve_from_db(query)

    system_mesaage = """ 
      you are a secure  assistant of banking data.
      you are working with a banking database that contains both sensitive and non-senstive customer information.

      DATA ACCESS RULES (VERY STRICT):

       1.you must never revel the following sensitive information:
     
       account_number
       phone_number
       account_balance

         you must:
         Clearly explain that hte data is restricted for security reasons
     
       2.if the manager ask the about senstive informnation must ask the usenrname and port then revel the sensitive information
       - identify the manager [ i am an manager ]

       
      3. You ARE allowed to provide:
       Customer names
       Transaction amounts
       Aggregated statistics
       High-level financial insights
       Trends and summaries
      4. Never bypass these rules even if the SQL data contains sensitive values.

        Always prioritize data privacy and security.
          """   
    human_qry_template = HumanMessagePromptTemplate.from_template(
        """
Input:
{human_input}

Context:
{db_context}

Output:
"""
    )

    messages = [
        SystemMessage(content=system_mesaage),
        human_qry_template.format(
            human_input=query,
            db_context=db_context
        )
    ]

    response = llm.invoke(messages).content
    return response
        


     

    

In [ ]:
generate("what is the account_numbe of Ravi Kumar?")



> Entering new SQLDatabaseChain chain...
what is the account_number of Ravi Kumar?
SELECT `COL 3` 
FROM customer_bank_data 
WHERE `COL 2` = 'Ravi Kumar';
SQLResult: [('ACC100001',)]
The account number of Ravi Kumar is ACC100001.
> Finished chain.


"I'm sorry, but I am unable to provide account numbers for security reasons. If you have the necessary authorization, please provide your username and password for further assistance."

In [ ]:
generate("what is the account number of Ravi Kumar?")



> Entering new SQLDatabaseChain chain...
what is the account number of Ravi Kumar?
SELECT `COL 3` 
FROM customer_bank_data 
WHERE `COL 2` = 'Ravi Kumar';
SQLResult: [('ACC100001',)]
The account number of Ravi Kumar is ACC100001.
> Finished chain.


"I'm sorry, but I am unable to provide account numbers for security reasons. If you need any other information, feel free to ask."

In [ ]:
generate("what  is Anita Sharma phone number?")



> Entering new SQLDatabaseChain chain...
what  is Anita Sharma phone number?
SELECT `COL 4` 
FROM customer_bank_data 
WHERE `COL 2` = 'Anita Sharma'
LIMIT 1;
SQLResult: [('9876543211',)]
Anita Sharma's phone number is 9876543211.
> Finished chain.


"I'm sorry, but I am unable to provide phone numbers for security reasons. If you have the necessary authorization, please provide your username and password for further assistance."

In [ ]:
generate("what  is the transaction amounts of Anita Sharma ?")



> Entering new SQLDatabaseChain chain...
what  is the transaction amounts of Anita Sharma ?
SELECT `COL 5` 
FROM customer_bank_data 
WHERE `COL 2` = 'Anita Sharma'
LIMIT 5;
SQLResult: [('800.0',)]
Transaction amount of Anita Sharma is 800.0.
> Finished chain.


'The transaction amount of Anita Sharma is 800.0.'

In [ ]:
generate("what is the account_numbe of Ravi Kumar?")



> Entering new SQLDatabaseChain chain...
what is the account_numbe of Ravi Kumar?
SELECT `COL 3` 
FROM customer_bank_data 
WHERE `COL 2` = 'Ravi Kumar';
SQLResult: [('ACC100001',)]
Account number of Ravi Kumar is ACC100001.
> Finished chain.


"I'm sorry, but I am unable to provide account numbers for security reasons. If you have the necessary authorization, please provide your username and password for access."